# The Anatomy of AI · Lecture 21 · Induction Heads — the notebook

A lighthouse keeper with a name no one has seen before, *Mr Quillon Barraclough*, appears twice in one short passage.
The first time, GPT-2 has no idea what comes after *Mr Qu*. The second time, it is almost certain. This lecture found the
two heads that do the trick, an **induction circuit**, first in GPT-2 and then in a tiny model we trained ourselves,
where we could watch them form. Here you redo it:

1. **GPT-2 copies**: the passage; a test with random tokens; the rule;
2. **our own model**: a 2-block transformer with no neurons, its copying head and the partner it needs, the
   in-context-learning score, how the two heads fit, and switching them off;
3. **watching it learn**: saved snapshots from training, the same heads forming; real names before random tokens; four
   seeds; the snap in longer runs;
4. **back in GPT-2**: its copying heads at the second name.

Our models were trained on Modal for the lecture (983 million FineWeb tokens; a free GPU would need days), so this
notebook downloads them from the course site: the final model in full, and thin snapshots from training. Blocks and heads
are counted from 1, as on the plates (papers count from 0). Every key number is checked against the lecture's own: a ✓
means you got what the lecture got.

It takes about **3 minutes** on free Colab with a GPU (*Runtime → Change runtime type → T4 GPU*).

In [ ]:
# Pinned installs. Free Colab already has torch; a machine without it gets the pinned version.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "requests==2.32.3", "pyarrow==18.1.0", "regex==2024.11.6")

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
torch.backends.cuda.matmul.allow_tf32 = False                  # plain float32 everywhere, as the lecture ran
T0 = time.time()

# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    number = isinstance(expected, (int, float)) and not isinstance(expected, bool)
    if number:
        ok = abs(float(value) - expected) <= tol
        whole = isinstance(expected, int) and float(value) == int(value)
        shown = f"{int(value):,}" if whole else f"{float(value):.6g}"
        said = f"{expected:,}" if isinstance(expected, int) else expected
    else:
        ok = value == expected
        shown, said = repr(value), repr(expected)
    extra = f" ± {tol:g}" if number and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(said)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path

## GPT-2 small, as Lecture 19 built it
Lecture 19's notebook in one cell: GPT-2's tokenizer from its own files, the weights read by hand, and `run()`, which
keeps every head's attention pattern and can switch heads off (`zero_heads=[(block, head), …]`, counted from 0).

In [ ]:
# GPT-2 small, exactly as Lecture 19 built it: no library model, only its files from Hugging Face (no key needed).
import regex as re
HF = "https://huggingface.co/openai-community/gpt2/resolve/main/"
fetch(HF + "vocab.json", "vocab.json", "GPT-2's token list")
fetch(HF + "merges.txt", "merges.txt", "GPT-2's 50,000 merges")
fetch(HF + "model.safetensors", "gpt2.safetensors", "GPT-2 small's weights")

# --- the tokenizer: bytes -> printable characters, a regex split, then the merges in rank order (Lecture 19, Plate VI)
VOCAB = json.load(open("vocab.json", encoding="utf-8"))
ID2TOK = {i: t for t, i in VOCAB.items()}
RANK = {tuple(m.split()): r for r, m in enumerate(l for l in open("merges.txt", encoding="utf-8").read().split("\n")[1:] if l)}
_bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
_cs = _bs + [256 + n for n in range(256 - len(_bs))]
_bs = _bs + [b for b in range(256) if b not in _bs]
B2U = dict(zip(_bs, map(chr, _cs))); U2B = {u: b for b, u in B2U.items()}
SPLIT = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")
_seen = {}

def bpe(piece):
    if piece not in _seen:
        w = list(piece)
        while len(w) > 1:
            r, i = min((RANK.get(p, 1 << 30), i) for i, p in enumerate(zip(w, w[1:])))
            if r == 1 << 30:
                break
            a, b, out, j = w[i], w[i + 1], [], 0
            while j < len(w):
                if j < len(w) - 1 and w[j] == a and w[j + 1] == b:
                    out.append(a + b); j += 2
                else:
                    out.append(w[j]); j += 1
            w = out
        _seen[piece] = w
    return _seen[piece]

def encode(text):
    return [VOCAB[t] for m in SPLIT.findall(text) for t in bpe("".join(B2U[b] for b in m.encode("utf-8")))]

def decode(ids):
    return bytes(U2B[ch] for i in ids for ch in ID2TOK[int(i)]).decode("utf-8", errors="replace")

# --- the weights: a safetensors file is an 8-byte length, a JSON table of contents, then the raw numbers
def read_safetensors(path):
    raw = open(path, "rb").read()
    n = int.from_bytes(raw[:8], "little")
    head = json.loads(raw[8:8 + n])
    out = {}
    for name, h in head.items():
        if name != "__metadata__" and not name.endswith((".attn.bias", ".attn.masked_bias")):   # masks, not weights
            a, b = h["data_offsets"]
            out[name] = torch.from_numpy(np.frombuffer(raw, dtype=np.float32, count=(b - a) // 4, offset=8 + n + a)
                                         .reshape(h["shape"]).copy()).to(DEV)
    return out

SD = read_safetensors("gpt2.safetensors")
WTE, WPE = SD["wte.weight"], SD["wpe.weight"]
print(f"GPT-2 small: {sum(v.numel() for v in SD.values()):,} weights")

# --- the forward pass (Lecture 19, Plates XIII-XXI; attention as Lecture 20 takes it apart)
def layer_norm(x, g, b, eps=1e-5):
    mu = x.mean(-1, keepdim=True)
    return (x - mu) / torch.sqrt(((x - mu) ** 2).mean(-1, keepdim=True) + eps) * g + b

def gelu(x):                                   # GPT-2's smooth gate (the tanh form)
    return 0.5 * x * (1 + torch.tanh(math.sqrt(2 / math.pi) * (x + 0.044715 * x ** 3)))

@torch.no_grad()
def run(ids, allow=None, zero_heads=()):
    """One pass of GPT-2 over a list of token ids. Returns every intermediate number in a dict.
    allow: optional (T, T) True/False table of which earlier words each word may look at (on top of 'no peeking ahead').
    zero_heads: (block, head) pairs, counted from 0, whose output is set to zero."""
    ids = torch.as_tensor(ids, device=DEV)
    T = len(ids)
    c = {"tok": WTE[ids], "pos": WPE[:T]}
    x = c["tok"] + c["pos"]
    mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=DEV))
    if allow is not None:
        mask = mask & allow
    keep = {k: [] for k in ("resid", "ln1", "q", "k", "v", "scores", "pattern", "z", "head_out", "attn_out", "mlp_post", "mlp_out")}
    for l in range(12):
        p = lambda n: SD[f"h.{l}.{n}"]
        keep["resid"].append(x)
        h = layer_norm(x, p("ln_1.weight"), p("ln_1.bias"))
        q, k, v = (h @ p("attn.c_attn.weight") + p("attn.c_attn.bias")).split(768, -1)
        q, k, v = (t.view(T, 12, 64).transpose(0, 1) for t in (q, k, v))            # 12 heads x T words x 64
        s = (q @ k.transpose(-1, -2) / 8).masked_fill(~mask, float("-inf"))          # 8 = the square root of 64
        a = torch.softmax(s, -1)
        z = a @ v
        ho = torch.einsum("htd,hdm->htm", z, p("attn.c_proj.weight").view(12, 64, 768))
        for zl, zh in zero_heads:
            if zl == l:
                ho[zh] = 0
        att = ho.sum(0) + p("attn.c_proj.bias")
        x = x + att
        m_in = layer_norm(x, p("ln_2.weight"), p("ln_2.bias")) @ p("mlp.c_fc.weight") + p("mlp.c_fc.bias")
        post = gelu(m_in)
        mo = post @ p("mlp.c_proj.weight") + p("mlp.c_proj.bias")
        x = x + mo
        for key, val in zip(("ln1", "q", "k", "v", "scores", "pattern", "z", "head_out", "attn_out", "mlp_post", "mlp_out"),
                            (h, q, k, v, s, a, z, ho, att, post, mo)):
            keep[key].append(val)
    keep["resid"].append(x)
    c.update({k: torch.stack(v) for k, v in keep.items()})
    c["final"] = layer_norm(x, SD["ln_f.weight"], SD["ln_f.bias"])
    c["logits"] = c["final"] @ WTE.T                 # the scores: the final total against every word's arrow
    return c

def lens(r):
    """The logit lens: read any running total through the final LayerNorm and the word table."""
    return layer_norm(r, SD["ln_f.weight"], SD["ln_f.bias"]) @ WTE.T

## Part 1 · GPT-2 copies a name

### 1. The second time is easy (Plate I)
For every token of the passage, GPT-2's chance for the token that really came next.

In [ ]:
PASSAGE = ("Every evening the lighthouse keeper, Mr Quillon Barraclough, climbed the hundred and twelve steps to light the "
           "lamp. In the village they liked to say that Mr Quillon Barraclough had not missed a single night in forty years.")
pas = [50256] + encode(PASSAGE)                       # <|endoftext|> first, as the lecture fed it
TOK = [decode([i]) for i in pas]
first, second = [i for i in range(len(pas)) if decode(pas[i:i + 2]) == " Quillon"]
cg = run(pas)
p_next = torch.softmax(cg["logits"][:-1], -1)[torch.arange(len(pas) - 1), torch.tensor(pas[1:])].tolist()
print("the name's pieces, first time and second time:")
for j in range(6):
    print(f"  {TOK[first + j]!r:8} {100 * p_next[first + j - 1]:7.2f}%   →   {100 * p_next[second + j - 1]:6.2f}%")
check("GPT-2's chance for every token of the passage (largest difference)", max(abs(a - b) for a, b in zip(p_next, [0.0004, 0.0011, 0.0197, 0.0, 0.0055, 0.0431, 0.0072, 0.0003, 0.0001, 0.0001, 0.0072, 0.2877, 0.9892, 0.8506, 0.0003, 0.1628, 0.0002, 0.0392, 0.0096, 0.1655, 0.2346, 0.0007, 0.6096, 0.0087, 0.2179, 0.0185, 0.4343, 0.0013, 0.0033, 0.0004, 0.6879, 0.0232, 0.2244, 0.0117, 0.3897, 0.9962, 0.0213, 0.9962, 0.9986, 0.9988, 0.1191, 0.0071, 0.0027, 0.3953, 0.1454, 0.042, 0.0502, 0.0006, 0.6708, 0.5051])), 0, tol=3e-4)
check("'illon', the first time", p_next[first], 0.0001, tol=5e-5)
check("'illon', the second time", p_next[second], 0.9962, tol=3e-4)

### 2. A test with no meaning at all (Plate II)
Maybe GPT-2 just knows something about lighthouse keepers. Remove all meaning: 50 random tokens, then the same 50 again,
20 times over (the lecture's own random tokens: NumPy's generator with seed 21). The first copy is pure guesswork. The
second need not be.

In [ ]:
rng = np.random.default_rng(21)
pos_loss = np.zeros(100)
attn = torch.zeros(12, 12, device=DEV)                  # each head's attention to "the token after the earlier copy"
back = torch.zeros(12, 12, device=DEV)                  # ... and to the token just before
REP21 = []
for _ in range(20):
    r = rng.integers(1000, 30000, size=50)
    ids = np.concatenate([[50256], r, r]).tolist(); REP21.append(ids)
    c = run(ids)
    lp = torch.log_softmax(c["logits"][:-1], -1).gather(-1, torch.tensor(ids[1:], device=DEV)[:, None])[:, 0]
    pos_loss += -lp.cpu().numpy() / 20
    i = torch.arange(51, 101, device=DEV)
    attn += c["pattern"][:, :, i, i - 49].mean(-1) / 20
    back += c["pattern"].diagonal(-1, dim1=-2, dim2=-1).mean(-1) / 20
loss1, loss2 = pos_loss[1:50].mean(), pos_loss[50:].mean()
print(f"loss on the first copy {loss1:.2f} nats; on the second {loss2:.2f}")
check("loss on the first copy", loss1, 12.7603, tol=0.01)
check("loss on the second copy", loss2, 0.4725, tol=0.01)
check("the loss at every position (largest difference)", float(np.abs(pos_loss - np.array([14.489, 14.958, 14.585, 14.741, 14.064, 13.812, 14.594, 13.739, 13.08, 13.552, 13.687, 13.317, 13.673, 12.648, 12.008, 12.442, 12.458, 12.375, 12.711, 12.65, 12.45, 13.055, 12.676, 12.168, 11.764, 11.407, 12.485, 12.75, 12.668, 12.351, 12.228, 11.798, 12.394, 12.29, 12.233, 11.987, 11.766, 13.84, 12.053, 12.411, 12.904, 11.924, 11.914, 12.732, 12.278, 12.69, 12.033, 12.123, 12.897, 11.889, 11.066, 3.325, 1.204, 1.871, 0.431, 0.543, 0.227, 0.247, 0.248, 0.223, 0.142, 0.083, 0.483, 0.157, 0.054, 0.113, 0.047, 0.074, 0.116, 0.063, 0.042, 0.017, 0.619, 0.485, 0.049, 0.034, 0.049, 0.031, 0.015, 0.063, 0.039, 0.027, 0.052, 0.02, 0.135, 0.028, 0.054, 0.101, 0.085, 0.066, 0.025, 0.131, 0.026, 0.029, 0.121, 0.029, 0.059, 0.175, 0.038, 0.265])).max()), 0, tol=0.01)
fig, ax = plt.subplots(figsize=(7, 2.6))
ax.plot(range(1, 101), pos_loss, color=INK, lw=1.4); ax.axvline(50.5, color=INK2, ls="--", lw=1)
ax.set_xlabel("position"); ax.set_ylabel("loss (nats)"); ax.set_title("50 random tokens, then the same 50 again", fontsize=10); plt.show()

### 3. The rule (Plate III)
In the second copy, the next token is whatever came after the *same* token the first time. The rule:
**[A][B] … [A] → [B]**. A head that does it attends, from each token of the second copy, to the token *after* that token's
earlier copy. Its **induction score** is the average share of attention it puts there. Which of GPT-2's 144 heads score
highly? And which look one token back?

In [ ]:
flat = sorted(((attn[l, h].item(), l, h) for l in range(12) for h in range(12)), reverse=True)
for v, l, h in flat[:5]:
    print(f"  block {l + 1:2}, head {h + 1:2}: induction score {v:.3f}")
check("the five copying heads (block, head, from 0)", [[l, h] for _, l, h in flat[:5]], [[int(x) for x in re.findall(r"\d+", n)] for n, _ in [['L5H5', 0.929], ['L6H9', 0.918], ['L7H10', 0.917], ['L5H1', 0.9], ['L7H2', 0.84]]])
check("their scores (largest difference)", max(abs(v - e) for (v, _, _), (_, e) in zip(flat[:5], [['L5H5', 0.929], ['L6H9', 0.918], ['L7H10', 0.917], ['L5H1', 0.9], ['L7H2', 0.84]])), 0, tol=3e-3)
l, h = divmod(int(back.argmax()), 12)
print(f"looks one token back most: block {l + 1}, head {h + 1} ({back[l, h]:.3f})")
check("the look-back head (block, head, from 0)", [l, h], [4, 11])
check("its score", back[l, h].item(), 0.987, tol=3e-3)

## Part 2 · Our own model

### 4. A model small enough to watch (Plate V)
The lecture trained its own transformers: GPT-2's tokens, 256 numbers per token, blocks of 8 heads of 32, **no neurons**
and no biases in attention, so every head reads from and writes to the running total directly. Trained on 983 million
FineWeb tokens (15,000 steps of 64 × 1,024 tokens; 55 minutes on one GPU). Two blocks are enough for the trick; one is not.

The model comes from the course site as *safetensors* files in bf16, exactly as training saved them, in two parts
(everything but the last table; the last table). Each file's SHA-256 fingerprint is checked first.

In [ ]:
import hashlib, re
SHA = {'l21_2block_a.safetensors': '069ef33515b81cff6f2129bdbed87fa4a2bccac2a017b166586b7bdb4fc1d78d', 'l21_2block_b.safetensors': 'a4f7314426d6afc9b060439cc994f38385c4c583813fc964fb8c321b3b3fb73b', 'l21_snap_fast.safetensors': 'd094e613ab0b9d15065094aef8e95060da0f50d897aace949e4afb24ac45a9c1', 'l21_snap_finals.safetensors': '1ec0687efe811464101a1d73d4322576289a596a98e1f9bb7bc0eecd96930971', 'l21_snap_long.safetensors': 'cb3dedd0def704fc414a7bc53660b568128b0d9fb8cd3805172ac0d68da4dd43', 'l21_snap_s0.safetensors': 'b51f7676a7d86b482e33380ccda7b839fda51a2de8a209ea31e2f3d8ed45a094', 'l20_1block_a.safetensors': 'dca5c5a8d50154fffda0ffd2f56e20ddd1a374945888766770c5231c9f78ed70', 'l20_1block_b.safetensors': 'b216b8c909fd2f9bf30ae9147c9304839ce7f5f70dcdd5a6bfe7fd6f20b98e24'}

def read_weights(path):
    """A safetensors file, any number format, as float32 tensors on the GPU (integers stay integers)."""
    raw = bytearray(open(path, "rb").read())
    n = int.from_bytes(raw[:8], "little")
    head = json.loads(raw[8:8 + n])
    kinds = {"BF16": torch.bfloat16, "F16": torch.float16, "F32": torch.float32, "I32": torch.int32}
    out = {}
    for name, h in head.items():
        if name != "__metadata__":
            a, b = h["data_offsets"]
            k = kinds[h["dtype"]]
            t = torch.frombuffer(raw, dtype=k, count=(b - a) // k.itemsize, offset=8 + n + a).reshape(h["shape"]).to(DEV)
            out[name] = t.long() if k == torch.int32 else t.float()
    return out

def load_files(url, files, what):
    sd = {}
    for fn in files:
        fetch(url + fn, fn, f"{what}: {fn}")
        assert hashlib.sha256(open(fn, "rb").read()).hexdigest() == SHA[fn], f"{fn} is not the published file"
        sd.update(read_weights(fn))
    return sd

def small_run(sd, ids, zero=()):
    """Our attention-only transformer on a batch of token-id lists (all the same length).
    Returns the scores (or None, for a snapshot without the last table) and each block's attention patterns.
    zero: (block, head) pairs, from 0, whose output is set to zero."""
    ids = torch.as_tensor(ids, device=DEV)
    ids = ids[None] if ids.dim() == 1 else ids
    B_, T_ = ids.shape
    x = sd["E.weight"][sd["row"][ids] if "row" in sd else ids] + sd["P.weight"][:T_]
    mask = torch.tril(torch.ones(T_, T_, dtype=torch.bool, device=DEV))
    pats = []
    for l in range(sum(k.startswith("qkv.") for k in sd)):
        h = layer_norm(x, sd[f"ln.{l}.weight"], sd[f"ln.{l}.bias"])
        q, k, v = (h @ sd[f"qkv.{l}.weight"].T).split(256, -1)
        q, k, v = (t.view(B_, T_, 8, 32).transpose(1, 2) for t in (q, k, v))
        a = torch.softmax((q @ k.transpose(-1, -2) / math.sqrt(32)).masked_fill(~mask, float("-inf")), -1)
        pats.append(a)
        if f"o.{l}.weight" in sd:
            z = a @ v
            for zl, zh in zero:
                if zl == l:
                    z[:, zh] = 0
            x = x + z.transpose(1, 2).reshape(B_, T_, 256) @ sd[f"o.{l}.weight"].T
    logits = layer_norm(x, sd["lnf.weight"], sd["lnf.bias"]) @ sd["U.weight"].T if "U.weight" in sd else None
    return logits, pats

M2 = load_files("https://glass-box-sigma.vercel.app/lectures/21-induction-heads/notebook/", ["l21_2block_a.safetensors", "l21_2block_b.safetensors"], "the 2-block model")
print(f"2-block model loaded, {sum(v.numel() for v in M2.values()):,} weights")
check("its weights", sum(v.numel() for v in M2.values()), 26519552)

### 5. The name, the second time (Plate VI)
Our model's chance for each piece of the name the second time, averaged over the five pieces after *Qu*, and where its
copying head, **block 2, head 5**, looks while reading the second name.

In [ ]:
def name_chance(logits):
    p = torch.softmax(logits[0, :-1], -1)[torch.arange(len(pas) - 1), torch.tensor(pas[1:])]
    return [p[i - 1].item() for i in range(second + 1, second + 6)]

lg, P2 = small_run(M2, pas)
nc = name_chance(lg)
print("the second name, piece by piece:", [f"{TOK[second + 1 + j]!r} {100 * v:.1f}%" for j, v in enumerate(nc)])
print(f"average over the five pieces: {100 * np.mean(nc):.1f}%")
check("our model's name copying", np.mean(nc), 0.4711, tol=0.01)
def pas_ind(P, h):      # block 2's attention, from each piece of the second name to the piece after its first copy
    return float(sum(P[1][0, h, second + j, first + j + 1] for j in range(5)) / 5)
ind = [pas_ind(P2, h) for h in range(8)]
print("block 2's heads, attention to 'the piece after the earlier copy':", np.round(ind, 3))
check("block 2, head 5 does it (from 0)", int(np.argmax(ind)), 4)
check("...with attention", ind[4], 0.6248, tol=0.02)
A_ = P2[1][0, 4].cpu().numpy()
fig, ax = plt.subplots(figsize=(5.4, 5.4))
ax.imshow(A_[24:45, :] ** 0.5, cmap="gray_r", vmin=0, vmax=1, aspect="auto")
for j in range(5):
    ax.add_patch(plt.Circle((first + j + 1, second + j - 24), 0.7, fill=False, color=INK, lw=1.4))
ax.set_yticks(range(21), TOK[24:45], fontsize=7); ax.set_xticks(range(0, len(pas), 4), TOK[::4], rotation=90, fontsize=7)
ax.set_title("Block 2, head 5, words 25–45 (rings: the piece after the earlier copy)", fontsize=9); plt.show()

### 6. Its partner (Plate VII)
To attend to *the piece after the earlier copy*, block 2 needs each token to carry a note saying which token came just
before it. That note is written by a **look-back head** in block 1: **block 1, head 1** attends one token back.

In [ ]:
def pas_back(P, h):
    T_ = P[0].shape[-1]
    return float(np.mean([P[0][0, h, t, t - 1].item() for t in range(2, T_)]))
bk = [pas_back(P2, h) for h in range(8)]
print("block 1's heads, attention one token back on the passage:", np.round(bk, 3))
check("block 1, head 1 looks back most (from 0)", int(np.argmax(bk)), 0)
check("...on the passage", bk[0], 0.3236, tol=0.02)

### 7. The in-context-learning score (Plate VIII)
Olsson et al. (2022) measure in-context learning without any special text: on ordinary text, how much better does the
model predict the 500th token of a passage than the 50th? The **ICL score** is the loss at token 500 minus the loss at
token 50 (each averaged over 11 positions); negative means the context helps. The lecture used 256 passages of 1,024
tokens from FineWeb's validation file, chosen with NumPy's generator (seed 1234).

In [ ]:
fetch("https://huggingface.co/datasets/kjj0/fineweb10B-gpt2/resolve/main/fineweb_val_000000.bin", "fineweb_val.bin",
      "FineWeb's validation tokens")
VAL = np.fromfile("fineweb_val.bin", dtype=np.uint16, offset=1024)[:20_000_000]
rng = np.random.default_rng(1234)
starts = rng.integers(0, len(VAL) - 1026, size=256)
REP = rng.integers(1000, 30000, size=(64, 256))                 # (the lecture's random tokens, drawn next)
SHORT = [50256] + list(rng.integers(1000, 30000, size=30)) * 2  # ... and its short repeated sequence
VX = torch.tensor(np.stack([VAL[s:s + 1025] for s in starts]).astype(np.int64), device=DEV)

def per_position_loss(sd, zero=()):
    out = torch.zeros(1024, device=DEV)
    back_ = None
    for b in range(0, 256, 8):
        lg_, pats = small_run(sd, VX[b:b + 8, :-1], zero)
        out += torch.nn.functional.cross_entropy(lg_.transpose(1, 2), VX[b:b + 8, 1:], reduction="none").sum(0)
        if b < 32:                                               # look-back scores on the first 32 passages
            s_ = pats[0].diagonal(-1, dim1=-2, dim2=-1).mean((0, 2))
            back_ = s_ if back_ is None else back_ + s_
    return out / 256, back_ / 4

t = time.time()
pp, back_text = per_position_loss(M2)
icl = (pp[495:506].mean() - pp[45:56].mean()).item()
print(f"256 passages in {time.time() - t:.0f} s: loss {pp.mean():.4f}; ICL score {icl:+.3f}")
check("our model's loss on FineWeb's validation passages", pp.mean().item(), 4.6105, tol=0.01)
check("its ICL score", icl, -0.255, tol=0.01)
print("look-back score on ordinary text, block 1's heads:", np.round(back_text.tolist(), 3))
check("block 1, head 1's look-back on ordinary text", back_text[0].item(), 0.276, tol=0.005)
fig, ax = plt.subplots(figsize=(6.5, 2.6))
ax.plot(range(1, 1025), pp.cpu(), color=INK, lw=1); ax.set_xscale("log")
ax.set_xlabel("position in the passage (log scale)"); ax.set_ylabel("loss (nats)"); ax.set_title("Later tokens are easier", fontsize=10); plt.show()

### 8. How the two heads fit (Plate XI)
Read it from the weights alone. Block 2, head 5 decides where to look by comparing its query with each token's key; a
key is read from the running total, *including what block 1 wrote there*. **K-composition** (Elhage et al., 2021)
measures how much the key side of block 2, head 5 (its QK matrix, `W_Q W_Kᵀ`) reads what a block-1 head writes (its OV
matrix, `W_V W_O`): the size of their product, over the product of their sizes. A random matrix of the same size and rank
gives the chance level.

In [ ]:
def mats(l, h):
    W = M2[f"qkv.{l}.weight"].T                                     # 256 x 768: query | key | value
    Wq, Wk, Wv = (W[:, 256 * p + 32 * h: 256 * p + 32 * (h + 1)] for p in range(3))
    Wo = M2[f"o.{l}.weight"].T[32 * h: 32 * (h + 1)]                # 32 x 256
    return Wq @ Wk.T, Wv @ Wo
qk, _ = mats(1, 4)
kc = [((qk @ mats(0, h)[1].T).norm() / (qk.norm() * mats(0, h)[1].norm())).item() for h in range(8)]
g = torch.Generator(device=DEV).manual_seed(0)
_, ov = mats(0, 0)
base = []
for _ in range(20):
    Rm = torch.randn(256, 32, device=DEV, generator=g) @ torch.randn(32, 256, device=DEV, generator=g)
    Rm = Rm / Rm.norm() * ov.norm()
    base.append(((qk @ Rm.T).norm() / (qk.norm() * Rm.norm())).item())
print("block 2, head 5's keys reading each block-1 head:", np.round(kc, 4))
print(f"with block 1, head 1: {kc[0]:.3f}; a random matrix: {np.mean(base):.4f} ± {np.std(base):.4f}  ({kc[0] / np.mean(base):.1f}× chance)")
check("K-composition with each block-1 head (largest difference)", max(abs(a - b) for a, b in zip(kc, [0.1559, 0.0791, 0.1274, 0.0658, 0.1049, 0.0545, 0.0583, 0.0667])), 0, tol=2e-3)
check("the random level", np.mean(base), 0.0627, tol=0.005)

### 9. Switch them off (Plate XII)
Zero each of the 16 heads in turn, and both circuit heads together, and measure the name copying again.

In [ ]:
off = {}
for l in range(2):
    for h in range(8):
        off[f"L{l}H{h}"] = np.mean(name_chance(small_run(M2, pas, zero=[(l, h)])[0]))
both = np.mean(name_chance(small_run(M2, pas, zero=[(0, 0), (1, 4)])[0]))
for l in range(2):
    print(f"block {l + 1}:", "  ".join(f"head {h + 1} {100 * off[f'L{l}H{h}']:4.1f}%" for h in range(8)))
print(f"intact {100 * np.mean(nc):.1f}%; block 1 head 1 off {100 * off['L0H0']:.2f}%; block 2 head 5 off {100 * off['L1H4']:.2f}%; both {100 * both:.2f}%")
check("name copying with each head off (largest difference)", max(abs(off[k] - v) for k, v in {'L0H0': 0.0116, 'L0H1': 0.4361, 'L0H2': 0.5389, 'L0H3': 0.5035, 'L0H4': 0.5101, 'L0H5': 0.4816, 'L0H6': 0.4928, 'L0H7': 0.4083, 'L1H0': 0.5139, 'L1H1': 0.4965, 'L1H2': 0.2668, 'L1H3': 0.4952, 'L1H4': 0.0024, 'L1H5': 0.5131, 'L1H6': 0.4617, 'L1H7': 0.405}.items()), 0, tol=0.015)
check("both circuit heads off", both, 0.0015, tol=0.005)

### 10. Two heads, not one (Plate XVI)
Lecture 20's 1-block model (from its notebook's files on the course site) has a look-back head too, a sharper one. But
with one block, no head comes after it to use the note it writes.

In [ ]:
M1 = load_files("https://glass-box-sigma.vercel.app/lectures/20-attention/notebook/", ["l20_1block_a.safetensors", "l20_1block_b.safetensors"], "the 1-block model")
print(f"1-block model loaded, {sum(v.numel() for v in M1.values()):,} weights")
pp1, back1 = per_position_loss(M1)
nc1 = name_chance(small_run(M1, pas)[0])
print(f"1 block: look-back head {back1.max():.3f} on ordinary text; name copying {100 * np.mean(nc1):.1f}%")
print(f"2 blocks: look-back head {back_text.max():.3f}; name copying {100 * np.mean(nc):.1f}%")
check("the 1-block model's look-back head", back1.max().item(), 0.321, tol=0.005)
check("the 1-block model's name copying", np.mean(nc1), 0.0112, tol=0.003)

## Part 3 · Watching it learn

### 11. Snapshots from training (Plates VI–VII)
The lecture saved the model during training. To redraw the attention patterns on the passage and on a short repeated
sequence (30 random tokens, then the same 30), a snapshot needs only the two blocks' attention weights and the rows of the
word table for the 75 tokens those two texts use: about a megabyte each. Here are seed 0's at 1,000, 3,000, 5,000, 7,500
and 10,000 steps (each step is 65,536 tokens), with the final model of Step 4 as step 15,000.

In [ ]:
def snapshots(fn, what):
    raw = load_files("https://glass-box-sigma.vercel.app/lectures/21-induction-heads/notebook/", [fn], what)
    rows = raw.pop("rows")
    row = torch.full((50257,), -1, dtype=torch.long, device=DEV); row[rows] = torch.arange(len(rows), device=DEV)
    snaps = {}
    for k, v in raw.items():
        key, name = k.split(".", 1)
        snaps.setdefault(key, {"row": row})[name] = v
    print(f"{what}: {', '.join(sorted(snaps, key=lambda k: (k.split('@')[0], int(k.split('@')[1]))))}")
    return snaps

S0 = snapshots("l21_snap_s0.safetensors", "seed 0's snapshots")
S0["2L-s0@15000"] = M2
steps = [1000, 3000, 5000, 7500, 10000, 15000]
watch = {}
for st in steps:
    _, P = small_run(S0[f"2L-s0@{st}"], pas)
    watch[st] = (pas_ind(P, 4), pas_back(P, 0), P[1][0, 4].cpu().numpy())
    print(f"step {st:6,}: block 2 head 5 → after the earlier copy {watch[st][0]:.3f};  block 1 head 1 → one back {watch[st][1]:.3f}")
check("block 2, head 5 through training (largest difference)", max(abs(watch[s][0] - {'1000': {'pas_ind': 0.0241, 'pas_back': 0.0643, 'short_ind': 0.0176, 'short_back': 0.0614, 'best_ind': 4, 'best_back': 3}, '3000': {'pas_ind': 0.1161, 'pas_back': 0.0695, 'short_ind': 0.0297, 'short_back': 0.0621, 'best_ind': 4, 'best_back': 0}, '5000': {'pas_ind': 0.2327, 'pas_back': 0.1121, 'short_ind': 0.0295, 'short_back': 0.096, 'best_ind': 4, 'best_back': 0}, '7500': {'pas_ind': 0.5514, 'pas_back': 0.2165, 'short_ind': 0.063, 'short_back': 0.1608, 'best_ind': 4, 'best_back': 0}, '10000': {'pas_ind': 0.5076, 'pas_back': 0.2848, 'short_ind': 0.1034, 'short_back': 0.2164, 'best_ind': 4, 'best_back': 0}, '15000': {'pas_ind': 0.6248, 'pas_back': 0.3236, 'short_ind': 0.1363, 'short_back': 0.2711, 'best_ind': 4, 'best_back': 0}}[str(s)]["pas_ind"]) for s in steps), 0, tol=0.02)
check("block 1, head 1 through training (largest difference)", max(abs(watch[s][1] - {'1000': {'pas_ind': 0.0241, 'pas_back': 0.0643, 'short_ind': 0.0176, 'short_back': 0.0614, 'best_ind': 4, 'best_back': 3}, '3000': {'pas_ind': 0.1161, 'pas_back': 0.0695, 'short_ind': 0.0297, 'short_back': 0.0621, 'best_ind': 4, 'best_back': 0}, '5000': {'pas_ind': 0.2327, 'pas_back': 0.1121, 'short_ind': 0.0295, 'short_back': 0.096, 'best_ind': 4, 'best_back': 0}, '7500': {'pas_ind': 0.5514, 'pas_back': 0.2165, 'short_ind': 0.063, 'short_back': 0.1608, 'best_ind': 4, 'best_back': 0}, '10000': {'pas_ind': 0.5076, 'pas_back': 0.2848, 'short_ind': 0.1034, 'short_back': 0.2164, 'best_ind': 4, 'best_back': 0}, '15000': {'pas_ind': 0.6248, 'pas_back': 0.3236, 'short_ind': 0.1363, 'short_back': 0.2711, 'best_ind': 4, 'best_back': 0}}[str(s)]["pas_back"]) for s in steps), 0, tol=0.02)
fig, ax = plt.subplots(1, 6, figsize=(12, 2.4))
for a, st in zip(ax, steps):
    a.imshow(watch[st][2][30:45, 4:20] ** 0.5, cmap="gray_r", vmin=0, vmax=1)
    for j in range(5):
        a.add_patch(plt.Circle((first + j + 1 - 4, second + j - 30), 0.6, fill=False, color=INK, lw=1))
    a.set_title(f"step {st:,}", fontsize=9); a.set_xticks([]); a.set_yticks([])
fig.suptitle("Block 2, head 5 on the second name, through training (rings: the piece after the earlier copy)", fontsize=10, y=1.04)
plt.show()

### 12. Real names before random tokens (Plate X)
At three times the learning rate, the same circuit forms sooner, and the snapshots show it doing the passage's real
name well before it does the meaningless repeated tokens.

In [ ]:
def short_ind(P, h):    # on the short sequence: from each token of the second copy to the token after its first copy
    return float(np.mean([P[1][0, h, i, i - 29].item() for i in range(31, 61)]))
def short_back(P, h):
    return float(np.mean([P[0][0, h, t, t - 1].item() for t in range(1, 61)]))
FAST = snapshots("l21_snap_fast.safetensors", "the 3× learning-rate run's snapshots")
fast = {}
for st in [3000, 4000, 5000, 7500, 15000]:
    sd = FAST[f"2L-s0-lr3e-3@{st}"]
    fast[st] = (pas_ind(small_run(sd, pas)[1], 4), short_ind(small_run(sd, SHORT)[1], 4))
    print(f"step {st:6,}: the passage's name {fast[st][0]:.3f};  random tokens {fast[st][1]:.3f}")
check("the name, through training (largest difference)", max(abs(fast[s][0] - {'3000': {'pas_ind': 0.2078, 'pas_back': 0.2, 'short_ind': 0.0278, 'short_back': 0.1623, 'best_ind': 4, 'best_back': 6}, '4000': {'pas_ind': 0.4368, 'pas_back': 0.3182, 'short_ind': 0.0453, 'short_back': 0.1924, 'best_ind': 3, 'best_back': 6}, '5000': {'pas_ind': 0.6653, 'pas_back': 0.3854, 'short_ind': 0.1059, 'short_back': 0.2444, 'best_ind': 4, 'best_back': 6}, '7500': {'pas_ind': 0.7609, 'pas_back': 0.8738, 'short_ind': 0.3209, 'short_back': 0.7253, 'best_ind': 4, 'best_back': 6}, '15000': {'pas_ind': 0.8391, 'pas_back': 0.9935, 'short_ind': 0.4507, 'short_back': 0.9577, 'best_ind': 4, 'best_back': 6}}[str(s)]["pas_ind"]) for s in fast), 0, tol=0.02)
check("random tokens, through training (largest difference)", max(abs(fast[s][1] - {'3000': {'pas_ind': 0.2078, 'pas_back': 0.2, 'short_ind': 0.0278, 'short_back': 0.1623, 'best_ind': 4, 'best_back': 6}, '4000': {'pas_ind': 0.4368, 'pas_back': 0.3182, 'short_ind': 0.0453, 'short_back': 0.1924, 'best_ind': 3, 'best_back': 6}, '5000': {'pas_ind': 0.6653, 'pas_back': 0.3854, 'short_ind': 0.1059, 'short_back': 0.2444, 'best_ind': 4, 'best_back': 6}, '7500': {'pas_ind': 0.7609, 'pas_back': 0.8738, 'short_ind': 0.3209, 'short_back': 0.7253, 'best_ind': 4, 'best_back': 6}, '15000': {'pas_ind': 0.8391, 'pas_back': 0.9935, 'short_ind': 0.4507, 'short_back': 0.9577, 'best_ind': 4, 'best_back': 6}}[str(s)]["short_ind"]) for s in fast), 0, tol=0.02)

### 13. Four seeds, and the learning rate (Plates XIV–XV)
Train again from other random starting weights and the same circuit appears, in different heads. Which block-1 head
looks back and which block-2 head copies, in each seed's final model? And at a third of the learning rate, by the end of
the same 15,000 steps, has it formed at all?

In [ ]:
FIN = snapshots("l21_snap_finals.safetensors", "other runs' final models")
FIN["2L-s0@15000"] = M2
found = {}
for name in ["2L-s0", "2L-s1", "2L-s2", "2L-s3", "2L-s0-lr3e-4"]:
    _, P = small_run(FIN[f"{name}@15000"], pas)
    i_ = [pas_ind(P, h) for h in range(8)]; b_ = [pas_back(P, h) for h in range(8)]
    found[name] = [int(np.argmax(b_)), int(np.argmax(i_))]
    print(f"{name:13} look-back: block 1, head {found[name][0] + 1} ({max(b_):.2f});  copying: block 2, head {found[name][1] + 1} ({max(i_):.2f})")
check("each seed's two heads (from 0)", [found[r] for r in ["2L-s0", "2L-s1", "2L-s2", "2L-s3"]], [{'2L-s0': [0, 4], '2L-s1': [4, 2], '2L-s2': [5, 0], '2L-s3': [7, 6], '2L-s0-lr3e-4': [0, 3], '2L-s0-lr3e-3': [6, 4], '2L-long-s0': [0, 2], '2L-long-s1': [4, 7], '2L-long-s2': [5, 3]}[r] for r in ["2L-s0", "2L-s1", "2L-s2", "2L-s3"]])
_, P = small_run(FIN["2L-s0-lr3e-4@15000"], pas)
slow = max(pas_ind(P, h) for h in range(8))
check("at a third of the learning rate, the best copying head", slow, 0.1261, tol=0.02)

### 14. The snap (Plate XVII)
Three runs were trained further, to 3.3 billion tokens. In them, somewhere near a billion tokens, the look-back head
snaps from loose to sharp and the random-token copying switches on, within 100–200 million tokens. Seed 0's long run,
around the snap, on the short random sequence (its copying head is block 2, head 3):

In [ ]:
LONG = snapshots("l21_snap_long.safetensors", "the long run's snapshots")
snap = {}
for st in [10000, 12500, 15000, 17500, 20000]:
    sd = LONG[f"2L-long-s0@{st}"]
    _, P = small_run(sd, SHORT)
    snap[st] = (short_back(P, 0), short_ind(P, 2))
    print(f"{st * 65536 / 1e9:4.2f} B tokens: look-back {snap[st][0]:.2f};  copying random tokens {snap[st][1]:.2f}")
check("the look-back head around the snap (largest difference)", max(abs(snap[s][0] - {'10000': {'pas_ind': 0.0826, 'pas_back': 0.2528, 'short_ind': 0.0378, 'short_back': 0.2017, 'best_ind': 4, 'best_back': 0}, '12500': {'pas_ind': 0.3272, 'pas_back': 0.3177, 'short_ind': 0.0866, 'short_back': 0.2783, 'best_ind': 4, 'best_back': 0}, '15000': {'pas_ind': 0.7191, 'pas_back': 0.8048, 'short_ind': 0.1817, 'short_back': 0.6794, 'best_ind': 0, 'best_back': 0}, '17500': {'pas_ind': 0.8391, 'pas_back': 0.9389, 'short_ind': 0.3437, 'short_back': 0.898, 'best_ind': 2, 'best_back': 0}, '20000': {'pas_ind': 0.8484, 'pas_back': 0.9885, 'short_ind': 0.4848, 'short_back': 0.9601, 'best_ind': 2, 'best_back': 0}}[str(s)]["short_back"]) for s in snap), 0, tol=0.02)
check("random-token copying around the snap (largest difference)", max(abs(snap[s][1] - {'10000': {'pas_ind': 0.0826, 'pas_back': 0.2528, 'short_ind': 0.0378, 'short_back': 0.2017, 'best_ind': 4, 'best_back': 0}, '12500': {'pas_ind': 0.3272, 'pas_back': 0.3177, 'short_ind': 0.0866, 'short_back': 0.2783, 'best_ind': 4, 'best_back': 0}, '15000': {'pas_ind': 0.7191, 'pas_back': 0.8048, 'short_ind': 0.1817, 'short_back': 0.6794, 'best_ind': 0, 'best_back': 0}, '17500': {'pas_ind': 0.8391, 'pas_back': 0.9389, 'short_ind': 0.3437, 'short_back': 0.898, 'best_ind': 2, 'best_back': 0}, '20000': {'pas_ind': 0.8484, 'pas_back': 0.9885, 'short_ind': 0.4848, 'short_back': 0.9601, 'best_ind': 2, 'best_back': 0}}[str(s)]["short_ind"]) for s in snap), 0, tol=0.02)
fig, ax = plt.subplots(figsize=(5.5, 2.6))
xs = [s * 65536 / 1e9 for s in snap]
ax.plot(xs, [v[0] for v in snap.values()], color=INK, marker="o", label="look-back (block 1, head 1)")
ax.plot(xs, [v[1] for v in snap.values()], color=INK2, marker="s", ls="--", label="copying (block 2, head 3)")
ax.set_xlabel("billions of tokens"); ax.set_ylabel("attention"); ax.legend(fontsize=8); plt.show()

## Part 4 · Back in GPT-2 (Plate XVIII)
GPT-2's copying heads from Step 3, at the second name: from each piece, where does block 6, head 6 look? And what if
GPT-2's six strongest copying heads are switched off together?

In [ ]:
for j in range(6):
    i = second + j
    a = cg["pattern"][5, 5, i]
    k = int(a.argmax())
    print(f"  reading {TOK[i]!r:8} it looks at {TOK[k]!r:8} (word {k}) with {100 * a[k]:.0f}%")
check("at the second ' Qu', block 6 head 6 looks at", TOK[int(cg['pattern'][5, 5, second].argmax())], [' Qu', 'illon', 9, 0.944][1])
check("...with", cg["pattern"][5, 5, second].max().item(), [' Qu', 'illon', 9, 0.944][3], tol=3e-3)
six = [tuple(int(x) for x in re.findall(r"\d+", n)) for n in ['L5H5', 'L6H9', 'L7H10', 'L5H1', 'L7H2', 'L10H1']]
cz = run(pas, zero_heads=six)
pz = torch.softmax(cz["logits"][:-1], -1)[torch.arange(len(pas) - 1), torch.tensor(pas[1:])].tolist()
print(f"six copying heads off: 'illon' the second time {100 * p_next[second]:.1f}% → {100 * pz[second]:.1f}%")
check("the passage with six heads off (largest difference)", max(abs(a - b) for a, b in zip(pz, [0.0004, 0.0005, 0.0203, 0.0001, 0.0406, 0.0057, 0.0008, 0.0, 0.0, 0.0, 0.0035, 0.0083, 0.8175, 0.5276, 0.0, 0.2492, 0.0, 0.0092, 0.0057, 0.0, 0.0292, 0.0, 0.0004, 0.0, 0.4484, 0.0004, 0.1608, 0.0, 0.0015, 0.0, 0.7971, 0.0007, 0.032, 0.9964, 0.5862, 0.1048, 0.1195, 0.9955, 0.9887, 0.6555, 0.0057, 0.0023, 0.0001, 0.0058, 0.0325, 0.0297, 0.0055, 0.0, 0.2716, 0.627])), 0, tol=3e-4)
rand_off = np.mean([-torch.log_softmax(run(ids, zero_heads=six)["logits"][:-1], -1).gather(-1, torch.tensor(ids[1:], device=DEV)[:, None])[50:, 0].mean().item()
                    for ids in [np.concatenate([[50256], r, r]).tolist() for r in [np.random.default_rng(1000 + b).integers(1000, 30000, size=50) for b in range(20)]]])
print(f"random tokens, second copy: {loss2:.2f} nats → {rand_off:.2f} with the six off")
check("random tokens' second copy with the six off", rand_off, 6.5963, tol=0.01)

## Your turn

Each exercise has a one-line hint. Worked solutions are at the end.

**1. The other block-2 head.** In Step 9, switching off block 2, head 3 also cut the name copying. By how much?
*Hint: `off["L1H2"]`; then look at its pattern on the passage.*

In [ ]:
# your code here

**2. Both heads off, on ordinary text.** With the two circuit heads off, how much worse is our model's loss on the 256
validation passages, and its ICL score?
*Hint: `per_position_loss(M2, zero=[(0, 0), (1, 4)])`.*

In [ ]:
# your code here

**3. A second partner.** Which other block-1 head does block 2, head 5's key side read most?
*Hint: the list `kc` of Step 8.*

In [ ]:
# your code here

**4. A name of your own.** Write a short passage with an invented name used twice and test GPT-2 and our model.
*Hint: reuse `run()`, `small_run()` and the chance of each next token.*

In [ ]:
# your code here

## Solutions

**1. The other block-2 head.**

In [ ]:
print(f"block 2, head 3 off: name copying {100 * off['L1H2']:.1f}% (intact {100 * np.mean(nc):.1f}%)")
check("name copying with block 2, head 3 off", off["L1H2"], 0.2668, tol=0.015)
print("its attention on the second name:", np.round([pas_ind(P2, 2)], 3), "(head 5:", round(pas_ind(P2, 4), 3), ")")

**2. Both heads off, on ordinary text.**

In [ ]:
ppz, _ = per_position_loss(M2, zero=[(0, 0), (1, 4)])
iclz = (ppz[495:506].mean() - ppz[45:56].mean()).item()
print(f"loss {pp.mean():.3f} → {ppz.mean():.3f};  ICL score {icl:+.3f} → {iclz:+.3f}")
check("loss with both heads off", ppz.mean().item(), 4.9532, tol=0.01)
check("ICL score with both heads off", iclz, -0.1582, tol=0.01)

**3. A second partner.**

In [ ]:
order = np.argsort(kc)[::-1]
print("block-1 heads by K-composition:", [(f"head {h + 1}", round(kc[h], 3)) for h in order[:3]])
check("the second strongest (from 0)", int(order[1]), int(np.argsort([0.1559, 0.0791, 0.1274, 0.0658, 0.1049, 0.0545, 0.0583, 0.0667])[::-1][1]))

**4. A name of your own.**

In [ ]:
mine = [50256] + encode("At the market I met Mrs Thessaly Ombrewick, who sells honey. Everyone in town buys honey from Mrs Thessaly Ombrewick.")
words = [decode([i]) for i in mine]
pg = torch.softmax(run(mine)["logits"][:-1], -1)[torch.arange(len(mine) - 1), torch.tensor(mine[1:])]
po = torch.softmax(small_run(M2, mine)[0][0, :-1], -1)[torch.arange(len(mine) - 1), torch.tensor(mine[1:])]
for i in range(1, len(mine)):
    if words[i] in words[:i] and i > 20:
        print(f"  {words[i]!r:10} GPT-2 {100 * pg[i - 1]:5.1f}%   ours {100 * po[i - 1]:5.1f}%")

In [ ]:
print(f"The whole notebook ran in {(time.time() - T0) / 60:.1f} minutes.")